**Group Assignment - Part A: Practical Text Pre-Processing (Q5)**

This notebook provides an independent implementation of an alternative tokenization technique using SpaCy, alongside a comprehensive comparative analysis against the methods used in Q1.

**Environment Setup**

**Environment Dependencies**

First, we install and load the SpaCy library along with its official pre-trained English core language model (`en_core_web_sm`).


In [1]:
# %pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
print("Environment setup and SpaCy model download complete.")

Environment setup and SpaCy model download complete.


**Q5: Alternative Approach Implementation**

**Step 5.1: Alternative Tokenization using SpaCy**

The text corpus from `Data_1.txt` is loaded into a string variable and processed through SpaCy's natural language pipeline to perform advanced tokenization.


In [1]:
import spacy

# Read text from the local file
with open("Data_1.txt", "r", encoding="utf-8") as file:
    corpus_text = file.read()

nlp = spacy.load("en_core_web_sm")
doc = nlp(corpus_text)

spacy_tokens = [token.text for token in doc]

print(f"SpaCy Tokenization [Count: {len(spacy_tokens)} tokens]:\n")
print(spacy_tokens)

SpaCy Tokenization [Count: 97 tokens]:

['Classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'In', 'basic', '\n', 'classification', 'tasks', ',', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs', ',', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined', 'in', 'advance', '.', 'The', 'basic', 'classification', 'task', 'has', 'a', 'number', 'of', 'interesting', 'variants', '.', 'For', 'example', ',', 'in', 'multiclass', 'classification', ',', 'each', 'instance', 'may', 'be', 'assigned', 'multiple', 'labels', ';', 'in', 'open', '-', 'class', 'classification', ',', 'the', 'set', 'of', 'labels', 'is', 'not', 'defined', 'in', 'advance', ';', 'and', 'in', 'sequence', 'classification', ',', 'a', 'list', 'of', 'inputs', 'are', 'jointly', 'classified', '.']


**Step 5.2: Compare and Contrast the Alternative Approach with Q1's Best Approach**

To evaluate operational behaviors directly, we contrast the structural token outputs of NLTK's `word_tokenize` and SpaCy's pipeline tokenization side-by-side.

In [2]:
from nltk.tokenize import word_tokenize

tokens_nltk = word_tokenize(corpus_text)

print(f"Method 1: NLTK word_tokenize -> Total Tokens: {len(tokens_nltk)}")
print(f"Method 2: SpaCy Tokenizer    -> Total Tokens: {len(spacy_tokens)}")

print("\n--- Trailing Punctuation Experiment (labels;) ---")
nltk_punct = [t for t in tokens_nltk if t in ['labels', ';']]
spacy_punct = [t for t in spacy_tokens if t in ['labels', ';']]
print("NLTK Output :", nltk_punct)
print("SpaCy Output:", spacy_punct)

print("\n--- Hyphenated Words Experiment (open-class) ---")
nltk_hyphen = [t for t in tokens_nltk if 'open-class' in t or t in ['open', '-', 'class']]
spacy_hyphen = [t for t in spacy_tokens if 'open-class' in t or t in ['open', '-', 'class']]
print("NLTK Output :", nltk_hyphen)
print("SpaCy Output:", spacy_hyphen)

print("\n--- Standard Alphabetic Word Experiment (isolation) ---")
nltk_normal = [t for t in tokens_nltk if t == 'isolation']
spacy_normal = [t for t in spacy_tokens if t == 'isolation']
print("NLTK Output :", nltk_normal)
print("SpaCy Output:", spacy_normal)

Method 1: NLTK word_tokenize -> Total Tokens: 94
Method 2: SpaCy Tokenizer    -> Total Tokens: 97

--- Trailing Punctuation Experiment (labels;) ---
NLTK Output : ['labels', 'labels', ';', 'labels', ';']
SpaCy Output: ['labels', 'labels', ';', 'labels', ';']

--- Hyphenated Words Experiment (open-class) ---
NLTK Output : ['class', 'open-class']
SpaCy Output: ['class', 'open', '-', 'class']

--- Standard Alphabetic Word Experiment (isolation) ---
NLTK Output : ['isolation']
SpaCy Output: ['isolation']


**Comparison Result Interpretation**

Across the corpus, NLTK's `word_tokenize` produced **94 tokens**, while SpaCy's pipeline tokenizer produced **97 tokens**. Both tokenizers behaved identically on standard alphabetic words (e.g. `isolation`) and on trailing punctuation attached to a word (e.g. `labels;` was correctly split into `labels` and `;` by both).

The difference comes from two specific behaviours:
- **Hyphenated compounds:** NLTK kept `open-class` as a single token, treating the hyphen as part of the word. SpaCy instead split it into three tokens — `open`, `-`, `class` — treating the hyphen as its own punctuation-like token.
- **Whitespace handling:** SpaCy tokenized the embedded newline character in the corpus (`\n`) as its own token, whereas NLTK silently discarded it as whitespace.

Together, these two behaviours account for the 3-token difference between the two outputs (2 extra tokens from splitting `open-class`, 1 extra token from the retained newline).

**Step 5.3: Why the Alternative Approach is Better, Worse, or Just Different**

Neither tokenizer is simply "better" — they differ by design intent, and which one is preferable depends on the downstream task.

**Where SpaCy is arguably better:** by splitting hyphenated words like `open-class` into three separate tokens (`open`, `-`, `class`), SpaCy produces finer, linguistically informed units. This granularity is useful for tasks such as POS tagging, dependency parsing, and named-entity recognition, where SpaCy's tokenizer is built to feed directly into these deeper pipeline stages.

**Where NLTK is arguably better:** for this assignment's use case — a Bag-of-Words / TF-IDF style Naive Bayes text classifier (Lab 8) — keeping `open-class` intact as a single token is more useful, since splitting it into `open` and `class` would dilute a meaningful compound term into two generic, less informative words that lose their combined meaning. NLTK is also lighter and faster, since it doesn't require loading a full statistical language pipeline just to tokenize text, unlike SpaCy which loads a trained model (`en_core_web_sm`) even for this simple task.

**Where SpaCy introduces a downside:** it tokenizes whitespace characters (such as the embedded newline in the corpus) as their own token, which NLTK silently discards. Left unfiltered, this adds noise to downstream feature extraction and would need an extra cleaning step before use in a classifier.

**Conclusion:** SpaCy is *different* rather than strictly better — it is more linguistically expressive and better suited to tasks needing deeper syntactic or semantic structure, at the cost of speed and simplicity. For the group's Naive Bayes classification task, NLTK's simpler, faster, compound-preserving tokenizer is the more practical choice.